# 第15回：Show & Tellと自社データへの橋渡し

**今日の問い：自社データで始めるなら、最初の小さな一歩は何か。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。`DEEP DIVE`は経験者や自習向けの発展です。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- モデルの目的・検証・結果・限界を短く説明し、再現可能に共有する
- 学習済みPipelineをjoblibで保存し、モデルカードを関数で生成する
- 適用領域と較正の観点から、使ってよい範囲と監視項目を決める

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- モデルカード：用途・データ・評価・限界をまとめた記録
- 適用領域：モデルを使ってよい対象と条件
- 永続化：学習済みモデルをファイルへ保存すること
- ドリフト：運用後に入力や関係が変わること
- 監視：運用後の入力や性能変化を確認すること

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


## 最初から再実行できるか

第14回Notebookを`Kernel`→`Restart Kernel and Run All Cells`で実行し、提出CSVが同じ手順で作れることを確認します。


In [ ]:
import pandas as pd
experiment_data = pd.read_csv(DATA / "compound_experiments.csv")
print("共有する候補")
print("データ件数:", len(experiment_data))
print("活性率:", round(experiment_data["active"].mean(), 3))
print("収率の中央値:", experiment_data["yield_pct"].median())


## 1人5分のShow & Tell

面白かった図 / 改善した実験 / 悪化したが学びがあった実験 / Copilotへの良かった聞き方 / 自社テーマへ持ち帰りたい考え方 のうち1つを選びます。完成度は競いません。


## 自社テーマ1枚シート

機密情報や実データは書かず、一般化した表現で埋めます。

| 項目 | 記入内容 |
|---|---|
| 利用者と判断 | 誰が何を決めるか |
| 予測時点 | いつ予測するか |
| 目的変数 | 何を予測するか |
| 説明変数候補 | その時点で得られる情報 |
| 使えない情報 | 未来情報、測定後情報、機密上使えない情報 |
| 評価方法 | 指標と分割単位 |
| 単純な基準 | 平均、最頻値、現在の判断方法など |
| 最初の実験 | 1〜2週間で試せる小さな範囲 |


## DEEP DIVE：モデルの永続化・モデルカード・適用領域

発表で終わらせず、再現・共有・安全な運用まで一歩進めます。


In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

data = pd.read_csv(DATA / "compound_experiments.csv")
feat = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X_tr, X_te, y_tr, y_te = train_test_split(data[feat], data["active"], test_size=0.25, random_state=42, stratify=data["active"])
final = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)).fit(X_tr, y_tr)
path = ROOT / "workspace" / "final_model.joblib"
joblib.dump(final, path)
reloaded = joblib.load(path)
assert np.array_equal(final.predict(X_te), reloaded.predict(X_te)), "保存前後で予測が一致しません"
print("保存し読み直しても同じ予測:", path)


### モデルカードを関数で作る


In [ ]:
from sklearn.metrics import f1_score

def build_model_card(name, estimator, X_valid, y_valid, notes) -> pd.DataFrame:
    "モデルの用途と評価をまとめた1枚のカードを作る。"
    pred = estimator.predict(X_valid)
    items = {
        "モデル名": name,
        "検証F1": round(f1_score(y_valid, pred), 3),
        "想定利用者": notes["利用者"],
        "支援する判断": notes["判断"],
        "既知の限界": notes["限界"],
        "使ってはいけない条件": notes["禁止"],
    }
    return pd.DataFrame({"項目": list(items), "内容": list(items.values())})

build_model_card("活性スクリーナ", reloaded, X_te, y_te, {
    "利用者": "実験担当者", "判断": "追試する候補の優先順位",
    "限界": "新規scaffoldでは精度低下の可能性", "禁止": "測定後の列を入力に使うこと",
})


### 適用領域：予測してよい範囲を数値化する

学習データから遠い試料は、予測を鵜呑みにせず要確認に回します。近傍距離で範囲外を仕分けます。


In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

train_filled = X_tr.fillna(X_tr.median())
scaler = StandardScaler().fit(train_filled)
nn = NearestNeighbors(n_neighbors=5).fit(scaler.transform(train_filled))
train_dist = nn.kneighbors(scaler.transform(train_filled))[0].mean(axis=1)
threshold = np.quantile(train_dist, 0.95)
valid_dist = nn.kneighbors(scaler.transform(X_te.fillna(X_tr.median())))[0].mean(axis=1)
out_of_domain = valid_dist > threshold
print(f"適用領域外と判定された検証試料: {int(out_of_domain.sum())} / {len(valid_dist)} 件")
print("範囲外は予測を鵜呑みにせず、要確認に回す運用が考えられる。")


## よくある誤り

- スコアだけを成果として示す
- 自社データの利用許可や来歴を省略する
- 本番投入を最初の試行にする

## SELF-STUDY（任意・30〜60分）

- 保存したPipelineを読み直し、同じ入力で同じ予測になるか検証する
- 適用領域スコアを閾値化し、範囲外の試料を要確認として仕分ける

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. このモデルは誰の何の判断を助けるか
2. 適用領域をどう数値化したか
3. 運用後に監視すべき指標は何か

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
